# Ridge paths in prediction–control geometry

This notebook reproduces the two-panel figure from scratch. The left panel sweeps noise variance $\sigma^2$ on a logarithmic grid at fixed ridge penalty; the right panel sweeps $\lambda$ (and the corresponding $\kappa$) on a logarithmic grid at fixed noise variance. Edit the parameter and layer cells, then rerun the plotting cell.

For $y=X\beta^\star+\xi$ and $\xi\sim\mathcal N(0,\sigma^2I)$, define $A_\lambda=(X^\top X+n\lambda I)^{-1}X^\top$. Conditional on the displayed design $X$,

$$\hat\beta_\lambda\mid X\sim\mathcal N\!\left(A_\lambda X\beta^\star,\;\sigma^2A_\lambda A_\lambda^\top\right).$$

In [ ]:
from dataclasses import replace
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

repo_root = Path.cwd().resolve()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
if not (repo_root / 'scripts' / 'plot_ridge_paths_in_geometry.py').exists():
    raise RuntimeError('Run this notebook from the repository root or notebooks/.')
sys.path.insert(0, str(repo_root))

from scripts.plot_ridge_paths_in_geometry import (
    PlotElements, PlotStyle, make_figure, make_lambda_only_figure,
)

## 1. Numerical experiment

The grids are logarithmic, while $\sigma^2=0$ is added separately to show the conditional mean. `sigma2_cloud_levels` and `lambda_cloud_levels` choose the exact Gaussian ellipses that are drawn; the solid paths still use the full grids. When `covariance_crossover=True`, the exact $\lambda_\times$ where the two conditional covariance eigenvalues cross is added automatically.

In [ ]:
experiment = dict(
    epsilon=0.04,
    beta_star=np.array([1.1, 0.65]),
    n=40,
    fixed_lambda=0.05,
    fixed_sigma2=0.5,
    sigma2_range=(1e-4, 4.0),
    lambda_range=(1e-4, 10.0),
    n_sigma2=81,
    n_lambda=101,
    sigma2_cloud_levels=(0.01, 0.1, 1.0, 4.0),
    lambda_cloud_levels=(1e-4, 0.01, 1.0, 10.0),
    n_cloud=250,
    seed=20260918,
)

## 2. Visual layers and layout

Every visual layer can be switched independently. The default uses analytic Gaussian ellipses without Monte Carlo points and treats the iso-error contours as a faint background. `stationary_highlights=True` marks the prediction-tangency and $z=0$ crossing of the highlighted realization path. Set `cloud_points=True` to verify the ellipses using sampled response noise.

In [ ]:
elements = PlotElements(
    data_ellipse=True,
    prediction_contours=True,
    control_contours=True,
    teacher=True,
    realization_paths=True,
    cloud_points=False,
    gaussian_ellipses=True,
    covariance_crossover=True,
    stationary_highlights=True,
    mean_paths=True,
    point_labels=True,
    arrows=True,
    panel_notes=True,
    legend=True,
)

style = PlotStyle(
    figsize=(13.1, 6.45),
    lambda_only_figsize=(8.2, 7.0),
    path_color='#5B2A86',
    mean_color='0.28',
    geometry_alpha=0.22,
    geometry_linewidth_scale=0.52,
    cloud_point_alpha=0.10,
    cloud_point_size=11.0,
    gaussian_coverage=0.68,
    gaussian_fill_alpha=0.085,
    gaussian_linewidth=2.1,
    path_linewidth=2.7,
    legend_columns=3,
)

## 3. Reproduce and export

The PDF uses embedded editable TrueType text (`pdf.fonttype = 42`). One CSV contains every path and optional Monte Carlo point; the second contains the exact Gaussian means, covariance entries, ellipse axes, and orientations.

In [ ]:
output_png = repo_root / 'figures' / 'geometry' / 'ridge_paths_iso_error_geometry_thin_background.png'
output_pdf = repo_root / 'figures' / 'geometry' / 'ridge_paths_iso_error_geometry_thin_background.pdf'
output_csv = repo_root / 'tables' / 'ridge_paths_iso_error_geometry.csv'
output_gaussian_csv = repo_root / 'tables' / 'ridge_gaussian_ellipses.csv'

fig = make_figure(
    **experiment,
    elements=elements,
    style=style,
    output=output_png,
    pdf_output=output_pdf,
    table_output=output_csv,
    gaussian_table_output=output_gaussian_csv,
    close=False,
)
plt.show()

## 4. Saved emphasis and lambda-only variants

The emphasis version restores the earlier iso-error boundary widths. The lambda-only export removes the left noise-scale panel and uses the thinner background boundaries.

In [ ]:
emphasis_style = replace(style, geometry_linewidth_scale=1.0)
make_figure(
    **experiment, elements=elements, style=emphasis_style,
    output=repo_root / 'figures' / 'geometry' / 'ridge_paths_iso_error_geometry_emphasis.png',
    pdf_output=repo_root / 'figures' / 'geometry' / 'ridge_paths_iso_error_geometry_emphasis.pdf',
    table_output=None, gaussian_table_output=None, close=True,
)

fig_lambda = make_lambda_only_figure(
    **experiment, elements=elements, style=style,
    output=repo_root / 'figures' / 'geometry' / 'ridge_paths_iso_error_geometry_lambda_only.png',
    pdf_output=repo_root / 'figures' / 'geometry' / 'ridge_paths_iso_error_geometry_lambda_only.pdf',
    table_output=None, gaussian_table_output=None, close=False,
)
plt.show()

## 5. A cleaner variant

This optional cell shows how to derive a reduced style without rewriting the full configuration. It does not overwrite the main figure.

In [ ]:
clean_elements = replace(
    elements,
    cloud_points=False,
    point_labels=False,
    panel_notes=False,
)
clean_style = replace(style, figsize=(12.0, 5.7), geometry_alpha=0.12)

fig_clean = make_figure(
    **experiment,
    elements=clean_elements,
    style=clean_style,
    output=None,
    pdf_output=None,
    table_output=None,
    gaussian_table_output=None,
    close=False,
)
plt.show()